# News Topic Classifier Using BERT

## Problem Statement & Objective
Fine-tune a transformer model (BERT) to classify news headlines into 4 topic categories: World, Sports, Business, Science/Technology.

## Dataset
AG News Dataset from Hugging Face Datasets (with synthetic fallback)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from datasets import Dataset, DatasetDict
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import torch

np.random.seed(42)
torch.manual_seed(42)
%matplotlib inline

## Dataset Loading

In [ ]:
def create_synthetic_ag_news():
    """Create synthetic dataset for demonstration if Hugging Face download fails"""
    print("Creating synthetic dataset for demonstration...")
    
    sample_texts = {
        0: [
            "International leaders meet for peace talks in Geneva",
            "United Nations discusses climate change initiatives",
            "European Union announces new trade agreement",
            "Prime minister visits foreign country on diplomatic mission",
            "Tensions rise as regional conflict continues"
        ],
        1: [
            "Championship game ends in dramatic overtime victory",
            "Olympic athlete breaks world record in 100m sprint",
            "Football team wins league championship for third year",
            "Tennis star claims Grand Slam title at Wimbledon",
            "Basketball player scores 50 points in playoff game"
        ],
        2: [
            "Stock market reaches all-time high after positive earnings",
            "Tech company announces massive quarterly profits",
            "New business partnership drives industry growth",
            "Central bank adjusts interest rates to curb inflation",
            "Startup raises $100 million in Series B funding"
        ],
        3: [
            "Scientists develop breakthrough in artificial intelligence",
            "New discovery in quantum computing revolutionizes technology",
            "Space agency successfully launches Mars rover",
            "Researchers find cure for rare genetic disorder",
            "Renewable energy technology achieves record efficiency"
        ]
    }
    
    train_data = []
    test_data = []
    
    for label, texts in sample_texts.items():
        for i, text in enumerate(texts * 100):
            train_data.append({"text": text, "label": label})
        for i, text in enumerate(texts * 20):
            test_data.append({"text": text, "label": label})
    
    np.random.shuffle(train_data)
    np.random.shuffle(test_data)
    
    return DatasetDict({
        "train": Dataset.from_list(train_data),
        "test": Dataset.from_list(test_data)
    })

try:
    from datasets import load_dataset as hf_load_dataset
    print("Attempting to download AG News from Hugging Face...")
    dataset = hf_load_dataset("ag_news", download_mode="force_redownload")
    print(f"Train samples: {len(dataset['train'])}")
    print(f"Test samples: {len(dataset['test'])}")
except Exception as e:
    print(f"Could not download dataset: {e}")
    print("Using synthetic dataset instead.")
    dataset = create_synthetic_ag_news()

label_names = {0: "World", 1: "Sports", 2: "Business", 3: "Science/Technology"}

In [ ]:
print("Sample data:")
for i in range(3):
    print(f"\nSample {i+1}:")
    print(f"Text: {dataset['train'][i]['text']}")
    print(f"Label: {dataset['train'][i]['label']} ({label_names[dataset['train'][i]['label']]})")

## Tokenization & Preprocessing

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(200))

## Model Development & Training

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

training_args = TrainingArguments(
    output_dir="./bert-news-classifier",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=50,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

## Evaluation

In [ ]:
eval_results = trainer.evaluate()
print(f"Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"F1-Score: {eval_results['eval_f1']:.4f}")

In [ ]:
predictions = trainer.predict(small_eval_dataset)
pred_labels = np.argmax(predictions.predictions, axis=-1)
true_labels = predictions.label_ids

print(classification_report(true_labels, pred_labels, target_names=list(label_names.values())))

In [ ]:
cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=list(label_names.values()), 
            yticklabels=list(label_names.values()))
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## Save Model

In [ ]:
model.save_pretrained("./bert-news-classifier-final")
tokenizer.save_pretrained("./bert-news-classifier-final")
print("Model and tokenizer saved successfully!")

## Final Summary / Insights

- Successfully fine-tuned BERT on either AG News or synthetic dataset
- Implemented proper error handling and fallback mechanism
- Model can classify news into 4 categories: World, Sports, Business, Science/Technology
- Model saved and ready for deployment